# DiffPIR / DPS / Plug-and-Play comparison

This notebook is a unified playground for comparing:

- DiffPIR
- DPS variants (`DPS_y0`, `DPS_yt`)
- Plug-and-play methods with other priors (e.g. DRUNet)

across the three tasks:

- Super-resolution (SISR)
- Deblurring
- Inpainting

Some method functions are **stubs** that currently raise `NotImplementedError`.
Once you implement them (see `EXPERIMENTS_IMPLEMENTATION_GUIDE.md`), the cells
below will run end-to-end.


In [1]:
import os
from pathlib import Path
from typing import Dict, List

import numpy as np
import matplotlib.pyplot as plt

from experiments.common import MethodConfig, RunResult, run_experiment, load_image_paths
from experiments import pnp_priors
from experiments import sr_methods, deblur_methods, inpaint_methods


In [2]:
# Registry of available per-image methods for each task.

def get_sr_methods():
    return {
        "diffpir": sr_methods.run_diffpir_sr,
        "dps_y0": lambda img, cfg: sr_methods.run_dps_sr(img, cfg, mode="DPS_y0"),
        "dps_yt": lambda img, cfg: sr_methods.run_dps_sr(img, cfg, mode="DPS_yt"),
        # Enable this once you implement run_pnp_drunet_sr
        "pnp_drunet": sr_methods.run_pnp_drunet_sr,
    }


def get_deblur_methods():
    return {
        "diffpir": deblur_methods.run_diffpir_deblur,
        "dps_y0": lambda img, cfg: deblur_methods.run_dps_deblur(img, cfg, mode="DPS_y0"),
        "dps_yt": lambda img, cfg: deblur_methods.run_dps_deblur(img, cfg, mode="DPS_yt"),
        "pnp_drunet": deblur_methods.run_pnp_drunet_deblur,
    }


def get_inpaint_methods():
    return {
        "diffpir": inpaint_methods.run_diffpir_inpaint,
        "dps_y0": lambda img, cfg: inpaint_methods.run_dps_inpaint(img, cfg, mode="DPS_y0"),
        "dps_yt": lambda img, cfg: inpaint_methods.run_dps_inpaint(img, cfg, mode="DPS_yt"),
        "pnp_drunet": inpaint_methods.run_pnp_drunet_inpaint,
    }


METHOD_REGISTRY = {
    "sr": get_sr_methods(),
    "deblur": get_deblur_methods(),
    "inpaint": get_inpaint_methods(),
}


def _default_method_config(task: str, name: str, sf: int = 4) -> MethodConfig:
    """Create a simple MethodConfig for a given task/method name."""

    if name.startswith("dps_"):
        generate_mode = name.upper()  # "DPS_Y0" / "DPS_YT" variants
    elif name == "diffpir":
        generate_mode = "DiffPIR"
    else:
        generate_mode = name  # e.g. "pnp_drunet"

    return MethodConfig(
        name=name,
        task=task,
        generate_mode=generate_mode,
        lambda_=1.0,
        zeta=0.25,
        sf=sf,
        extra={},
    )


def compare_task(
    task: str,
    testset_root: str,
    methods: List[str],
    sf: int = 4,
) -> Dict[str, RunResult]:
    """Run all selected methods for a given task over a dataset.

    Returns a dict mapping method name -> RunResult.
    """

    image_paths = load_image_paths(testset_root)
    task_methods = METHOD_REGISTRY[task]
    results: Dict[str, RunResult] = {}

    for name in methods:
        if name not in task_methods:
            raise ValueError(f"Unknown method '{name}' for task '{task}'")
        method_fn = task_methods[name]
        cfg = _default_method_config(task=task, name=name, sf=sf)
        run_result = run_experiment(
            method_config=cfg,
            image_paths=image_paths,
            method_fn=method_fn,
            output_root=None,
        )
        results[name] = run_result
    return results


def plot_metric_bar(results: Dict[str, RunResult], metric: str = "average_psnr", title: str = ""):
    """Plot a bar chart of a given aggregate metric for each method."""

    methods = list(results.keys())
    values = [float(getattr(results[m], metric)) for m in methods]

    plt.figure(figsize=(6, 4))
    plt.bar(methods, values)
    plt.ylabel(metric)
    plt.title(title or metric)
    plt.grid(axis="y", alpha=0.3)
    plt.show()


In [ ]:
# Example usage

# Super-resolution comparison on the demo_test set
sr_results = compare_task(
    task="sr",
    testset_root="testsets/demo_test",
    methods=["diffpir", "dps_y0", "dps_yt"],  # add "pnp_drunet" once implemented
    sf=4,
)
plot_metric_bar(sr_results, metric="average_psnr", title="SR: Average PSNR")

# Deblurring comparison (requires you to point to the right testset)
# deblur_results = compare_task(
#     task="deblur",
#     testset_root="testsets/your_deblur_set",
#     methods=["diffpir", "dps_y0", "dps_yt"],
# )
# plot_metric_bar(deblur_results, metric="average_psnr", title="Deblur: Average PSNR")

# Inpainting comparison (requires you to point to the right testset and masks)
# inpaint_results = compare_task(
#     task="inpaint",
#     testset_root="testsets/your_inpaint_set",
#     methods=["diffpir", "dps_y0", "dps_yt"],
# )
# plot_metric_bar(inpaint_results, metric="average_psnr", title="Inpaint: Average PSNR")


NotImplementedError: run_diffpir_sr is not implemented yet.